<a href="https://colab.research.google.com/github/commertech-official/devops-start/blob/main/work_drive_full_deepseek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets

# ============================================================
# ПАРАМЕТРЫ ДВИГАТЕЛЯ
# ============================================================
p = 2
R_s = 5.0
R_r = 3.0
L_s = 0.3
L_r = 0.3
L_m = 0.28
J = 0.01

sigma = 1 - (L_m**2) / (L_s * L_r)
K_r = L_m / L_r
T_r = L_r / R_r   # постоянная времени ротора

# ============================================================
# ПОЛНАЯ МОДЕЛЬ АСИНХРОННОГО ДВИГАТЕЛЯ В d-q
# ============================================================
def motor_model(y, t, U_sd, U_sq, omega_k, M_load_func):
    """
    y = [I_sd, I_sq, Psi_rd, Psi_rq, omega_m]

    Уравнения:
    dI_sd/dt = (1/(sigma*L_s)) * (U_sd - R_s*I_sd + omega_k*sigma*L_s*I_sq + (K_r/T_r)*Psi_rd + K_r*p*omega_m*Psi_rq)
    dI_sq/dt = (1/(sigma*L_s)) * (U_sq - R_s*I_sq - omega_k*sigma*L_s*I_sd + (K_r/T_r)*Psi_rq - K_r*p*omega_m*Psi_rd)
    dPsi_rd/dt = (L_m/T_r)*I_sd - (1/T_r)*Psi_rd + (omega_k - p*omega_m)*Psi_rq
    dPsi_rq/dt = (L_m/T_r)*I_sq - (1/T_r)*Psi_rq - (omega_k - p*omega_m)*Psi_rd
    domega_m/dt = (M - M_load) / J
    """
    I_sd, I_sq, Psi_rd, Psi_rq, omega_m = y

    # Момент
    M = 1.5 * p * (Psi_rd * I_sq - Psi_rq * I_sd)

    # Нагрузка
    M_load = M_load_func(omega_m)

    # Уравнения токов статора
    dI_sd = (1/(sigma*L_s)) * (U_sd - R_s*I_sd + omega_k*sigma*L_s*I_sq
                                + (K_r/T_r)*Psi_rd + K_r*p*omega_m*Psi_rq)
    dI_sq = (1/(sigma*L_s)) * (U_sq - R_s*I_sq - omega_k*sigma*L_s*I_sd
                                + (K_r/T_r)*Psi_rq - K_r*p*omega_m*Psi_rd)

    # Уравнения потокосцеплений ротора
    dPsi_rd = (L_m/T_r)*I_sd - (1/T_r)*Psi_rd + (omega_k - p*omega_m)*Psi_rq
    dPsi_rq = (L_m/T_r)*I_sq - (1/T_r)*Psi_rq - (omega_k - p*omega_m)*Psi_rd

    # Уравнение движения
    domega_m = (M - M_load) / J

    return [dI_sd, dI_sq, dPsi_rd, dPsi_rq, domega_m]

# ============================================================
# ФУНКЦИЯ НАГРУЗКИ
# ============================================================
def M_load_func(omega_m, k_load=0.0005):
    """Вентиляторная нагрузка: M_load = k * ω²"""
    return k_load * omega_m**2

# ============================================================
# СИМУЛЯЦИЯ С ОРИЕНТАЦИЕЙ ПО ПОЛЮ РОТОРА
# ============================================================
def simulate_foc(I_sd_ref, I_sq_ref, t_end=3.0):
    """
    Симуляция с ориентацией по полю ротора (FOC).
    Предполагаем, что преобразователь идеально поддерживает I_sd и I_sq.
    """
    t = np.linspace(0, t_end, 3000)

    # Упрощённая модель: токи мгновенно следуют за заданиями
    I_sd = np.full_like(t, I_sd_ref)
    I_sq = np.full_like(t, I_sq_ref)

    # Потокосцепление ротора (установившееся)
    Psi_rd = L_m * I_sd_ref
    Psi_rq = 0

    # Момент
    M = 1.5 * p * Psi_rd * I_sq

    # Динамика скорости
    omega_m = np.zeros_like(t)
    for i in range(1, len(t)):
        M_load = M_load_func(omega_m[i-1])
        domega = (M[i] - M_load) / J * (t[1] - t[0])
        omega_m[i] = omega_m[i-1] + domega

    rpm = omega_m * 60 / (2 * np.pi)

    return t, rpm, M, Psi_rd, I_sd, I_sq

# ============================================================
# ИНТЕРАКТИВНЫЙ ГРАФИК
# ============================================================
def plot_foc(I_sd=5.0, I_sq=5.0):
    t, rpm, M, Psi_rd, I_sd_arr, I_sq_arr = simulate_foc(I_sd, I_sq)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Скорость
    axes[0, 0].plot(t, rpm, 'b-', linewidth=2)
    axes[0, 0].set_xlabel('Время, с')
    axes[0, 0].set_ylabel('Скорость, об/мин')
    axes[0, 0].set_title(f'Скорость вращения')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].axhline(y=rpm[-1], color='r', linestyle='--',
                        label=f'Установившаяся: {rpm[-1]:.0f} об/мин')
    axes[0, 0].legend()

    # Момент
    axes[0, 1].plot(t, M, 'g-', linewidth=2)
    axes[0, 1].set_xlabel('Время, с')
    axes[0, 1].set_ylabel('Момент, Н·м')
    axes[0, 1].set_title(f'Электромагнитный момент')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].axhline(y=M[0], color='r', linestyle='--',
                        label=f'M = {M[0]:.3f} Н·м')
    axes[0, 1].legend()

    # Токи Id и Iq
    axes[1, 0].plot(t, I_sd_arr, 'r-', linewidth=2, label='Id (намагничивание)')
    axes[1, 0].plot(t, I_sq_arr, 'b-', linewidth=2, label='Iq (момент)')
    axes[1, 0].set_xlabel('Время, с')
    axes[1, 0].set_ylabel('Ток, А')
    axes[1, 0].set_title('Токи Id и Iq')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()

    # Потокосцепление ротора
    axes[1, 1].plot(t, np.full_like(t, Psi_rd), 'm-', linewidth=2)
    axes[1, 1].set_xlabel('Время, с')
    axes[1, 1].set_ylabel('Ψ_rd, Вб')
    axes[1, 1].set_title(f'Потокосцепление ротора: Ψ_rd = {Psi_rd:.3f} Вб')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Вывод численных результатов
    print("="*60)
    print(f"Id = {I_sd:.1f} А, Iq = {I_sq:.1f} А")
    print(f"Момент: M = {M[0]:.4f} Н·м")
    print(f"Потокосцепление ротора: Ψ_rd = {Psi_rd:.4f} Вб")
    print(f"Установившаяся скорость: {rpm[-1]:.0f} об/мин")
    print("="*60)

# Запуск
interact(plot_foc,
         I_sd=FloatSlider(min=0, max=10, step=0.5, value=5.0,
                          description='Id (А):'),
         I_sq=FloatSlider(min=-10, max=10, step=0.5, value=5.0,
                          description='Iq (А):'))

interactive(children=(FloatSlider(value=5.0, description='Id (А):', max=10.0, step=0.5), FloatSlider(value=5.0…

<function __main__.plot_foc(I_sd=5.0, I_sq=5.0)>